In [1]:
%load_ext autoreload
%autoreload 2
# these packages cannot be autoreloaded
%aimport -modal
%aimport -modal_proto
%aimport -synchronicity

import sys, time
from pathlib import Path
sys.path.append(str(Path().resolve().parent / "src3"))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from vibrations_pipeline import process_vibrations_modal, volume, VOLUME_PATH, app

In [3]:
BASE_SAMPLE_DIR = Path('/home/ethantu/workspace/good-vibrations/experiment-20/data/samples')
sample_dir = BASE_SAMPLE_DIR / '000000'

In [4]:
# upload — force=True overwrites if already present
t0 = time.perf_counter()
with volume.batch_upload(force=True) as batch:
    batch.put_file(sample_dir / 'inputs/00_raw_vibrations.npy', f"{sample_dir.name}/inputs/00_raw_vibrations.npy")
    batch.put_file(sample_dir / 'metadata.jsonl',               f"{sample_dir.name}/metadata.jsonl")
t_upload = time.perf_counter() - t0
print(f"upload: {t_upload:.1f}s")

upload: 2.5s


In [ ]:
results = {}
with app.run():
    # warm up the container before benchmarking so cold start doesn't skew results
    process_vibrations_modal.remote(sample_dir.name, verbose=0)

    for mode in ["sequential", "batched"]:
        # batched uses PCLK_BATCH_SIZE=100 (safe on any GPU); sequential uses pclk_batch_size kwarg
        env = {"PCLK_MODE": mode, "PCLK_BATCH_SIZE": "100"}
        fn = process_vibrations_modal.with_options(env=env)
        t0 = time.perf_counter()
        fn.remote(sample_dir.name, verbose=2)
        results[mode] = time.perf_counter() - t0
        print(f"[{mode}] total: {results[mode]:.1f}s")

print(f"\n--- pclk benchmark ---")
for mode, t in results.items():
    print(f"  {mode:12s}: {t:.1f}s")
print(f"  speedup: {results['sequential']/results['batched']:.1f}x")

In [ ]:
# 3. download outputs back to local
ESSENTIAL_FILES = [
    "inputs/01_raw_shifts.npy",
    "inputs/03_fft_shifts.npz",
    "inputs/04_recovered_audio.wav",
    "inputs/05_processed_fft.npy",
]
SYMLINKS = [
    ("recovered_audio.wav", "inputs/04_recovered_audio.wav"),
    ("X.npy",               "inputs/05_processed_fft.npy"),
]

t0 = time.perf_counter()
for rel in ESSENTIAL_FILES:
    local_path = sample_dir / rel
    local_path.parent.mkdir(parents=True, exist_ok=True)
    with open(local_path, "wb") as f:
        for chunk in volume.read_file(f"{sample_dir.name}/{rel}"):
            f.write(chunk)

for dst_rel, src_rel in SYMLINKS:
    dst = sample_dir / dst_rel
    src = sample_dir / src_rel
    if dst.exists() or dst.is_symlink(): dst.unlink()
    dst.symlink_to(src.relative_to(dst.parent))

t_download = time.perf_counter() - t0
print(f"download: {t_download:.1f}s")

download: 5.3s


In [ ]:
print(f"\n--- full pipeline summary (batched pclk) ---")
print(f"upload:                {t_upload:.1f}s")
print(f"modal (pclk+pipeline): {results['batched']:.1f}s")
print(f"download:              {t_download:.1f}s")
print(f"total:                 {t_upload + results['batched'] + t_download:.1f}s")